## How have team strategies evolved across seasons?
Compare powerplay vs death over run rates by season. Show whether teams have become more aggressive.

In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

matches = pd.read_csv("matches.csv")
deliveries = pd.read_csv("deliveries.csv")

# Apply cleaning
matches['city'].fillna('Unknown', inplace=True)
matches['player_of_match'].fillna('Unknown', inplace=True)
matches['winner'].fillna('No Result', inplace=True)
matches['season'] = matches['season'].replace({
    '2007/08': '2008', '2009/10': '2010', '2020/21': '2020'
}).astype(int)

name_map = {
    'Delhi Daredevils': 'Delhi Capitals',
    'Kings XI Punjab': 'Punjab Kings',
    'Royal Challengers Bangalore': 'Royal Challengers Bengaluru',
    'Deccan Chargers': 'Sunrisers Hyderabad',
    'Rising Pune Supergiant':'Rising Pune Supergiants'
}
for col in ['team1','team2','winner','toss_winner']:
    matches[col] = matches[col].replace(name_map)

deliveries = deliveries[deliveries['inning'] <= 2]  # remove super overs

C:\Users\hp\AppData\Local\Temp\ipykernel_4380\1044635566.py:10: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  matches['city'].fillna('Unknown', inplace=True)
C:\Users\hp\AppData\Local\Temp\ipykernel_4380\1044635566.py:11: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, w

In [5]:
# ── Q3 — Strategy Evolution (05_analysis_strategy.ipynb) ──
# Merge to get season info into deliveries
del_with_season = deliveries.merge(matches[['id','season']], left_on='match_id', right_on='id')

# Powerplay = overs 1-6, Death = overs 17-20
powerplay = del_with_season[del_with_season['over'] <= 6]
death = del_with_season[del_with_season['over'] >= 17]

pp_by_season = powerplay.groupby('season')['total_runs'].mean().reset_index()
pp_by_season.columns = ['season', 'avg_pp_runs_per_ball']

death_by_season = death.groupby('season')['total_runs'].mean().reset_index()
death_by_season.columns = ['season', 'avg_death_runs_per_ball']

print(pp_by_season)
print(death_by_season)

    season  avg_pp_runs_per_ball
0     2008              1.209857
1     2009              1.111045
2     2010              1.237960
3     2011              1.126203
4     2012              1.145477
5     2013              1.105375
6     2014              1.198659
7     2015              1.224606
8     2016              1.220580
9     2017              1.306956
10    2018              1.331019
11    2019              1.287805
12    2020              1.219115
13    2021              1.226223
14    2022              1.235440
15    2023              1.368281
16    2024              1.478885
    season  avg_death_runs_per_ball
0     2008                 1.673790
1     2009                 1.507910
2     2010                 1.613951
3     2011                 1.545062
4     2012                 1.611845
5     2013                 1.620821
6     2014                 1.687060
7     2015                 1.646008
8     2016                 1.681687
9     2017                 1.607810
10    2018

Strategy Evolution

Powerplay scoring has jumped from 7.26 runs/over in 2008 to 8.87 in 2024 which is a 22% increase that reflects the rise of aggressive openers and impact player rules
Death over scoring climbed from 10.04 to 10.87 runs/over in the same period which is more more modest, suggesting death bowling has improved in parallel with batting
The biggest jumps came in 2017 and then 2023–24, coinciding with rule changes and a new generation of power-hitters entering the IPL

## Toss advantage, does it actually matter?
Toss win → match win rate across venues.

In [6]:
# ── Q4 — Toss Analysis 
toss = matches[matches['winner'] != 'No Result'].copy()
toss['toss_won_match'] = toss['toss_winner'] == toss['winner']
toss_summary = toss['toss_won_match'].value_counts(normalize=True)
print("Overall toss win -> match win rate:")
print(toss_summary)

# By venue
venue_toss = toss.groupby('venue')['toss_won_match'].agg(['mean','count']).reset_index()
venue_toss.columns = ['venue','toss_win_rate','matches']
venue_toss = venue_toss[venue_toss['matches'] >= 20]  # only venues with enough data
print(venue_toss.sort_values('toss_win_rate', ascending=False).head(10))

Overall toss win -> match win rate:
toss_won_match
True     0.508257
False    0.491743
Name: proportion, dtype: float64
                                      venue  toss_win_rate  matches
30  Maharashtra Cricket Association Stadium       0.636364       22
49                  Sharjah Cricket Stadium       0.571429       28
14                             Eden Gardens       0.558442       77
23                    M Chinnaswamy Stadium       0.555556       63
56                 Wankhede Stadium, Mumbai       0.555556       45
10       Dr DY Patil Sports Academy, Mumbai       0.550000       20
46                   Sawai Mansingh Stadium       0.531915       47
16                         Feroz Shah Kotla       0.525424       59
27          MA Chidambaram Stadium, Chepauk       0.520833       48
50                     Sheikh Zayed Stadium       0.517241       29


Toss Advantage

Overall, winning the toss leads to a match win only 50.8% of the time, barely better than a coin flip, debunking the myth that the toss is decisive
But venue matters enormously: Maharashtra Cricket Association Stadium shows 63.6% toss-to-win conversion that is nearly two-thirds of matches go to the toss winner there.
Eden Gardens (55.8%), Chinnaswamy (55.6%), and Wankhede (55.6%) consistently favour the toss winner which means teams should factor venue into toss strategy.

## Who are the "big match" underperformers?
Compare regular avg vs knockout/final performance. High average, low final performance.

In [7]:
# ── Q5 — Consistency: Regular vs Knockout ──
knockout_types = ['Final', 'Semi Final', 'Qualifier 1', 'Qualifier 2', 'Eliminator']
knockout_matches = matches[matches['match_type'].isin(knockout_types)]['id']
regular_matches = matches[~matches['match_type'].isin(knockout_types)]['id']

del_knockout = deliveries[deliveries['match_id'].isin(knockout_matches)]
del_regular = deliveries[deliveries['match_id'].isin(regular_matches)]

def batting_avg(df):
    return df.groupby('batter').agg(
        runs=('batsman_runs','sum'),
        innings=('match_id','nunique')
    ).assign(avg=lambda x: x['runs']/x['innings'])

ko_avg = batting_avg(del_knockout).rename(columns={'avg':'knockout_avg'})
reg_avg = batting_avg(del_regular).rename(columns={'avg':'regular_avg'})

comparison = reg_avg[['regular_avg']].join(ko_avg[['knockout_avg']]).dropna()
comparison = comparison[(comparison.index.map(lambda x: del_regular[del_regular['batter']==x]['match_id'].nunique() >= 20))]
comparison['diff'] = comparison['regular_avg'] - comparison['knockout_avg']
print("Biggest underperformers in knockouts:")
print(comparison.sort_values('diff', ascending=False).head(10))

Biggest underperformers in knockouts:
                 regular_avg  knockout_avg       diff
batter                                               
N Pooran           24.915493      0.000000  24.915493
E Lewis            26.080000      2.000000  24.080000
JP Duminy          27.671233      4.500000  23.171233
TM Head            33.545455     11.333333  22.212121
KC Sangakkara      25.134328      3.000000  22.134328
ML Hayden          36.413793     17.000000  19.413793
Abhishek Sharma    23.448276      5.666667  17.781609
A Badoni           18.617647      1.000000  17.617647
EJG Morgan         19.661972      3.000000  16.661972
C de Grandhomme    15.150000      0.000000  15.150000


Knockout Underperformers

Multiple high-average players show dramatic drops in knockout matches, proving that regular season form doesn't always translate when stakes are highest.
The pattern suggests pressure and unfamiliarity with opposition analysis in knockouts affects even experienced international players
Teams building squads should weight knockout performance more heavily than league-stage averages, a player with lower overall average but consistent knockout numbers is worth more.